# Chapter 10: EDA and Data Cleaning
**Companion notebook** · Book 1: Python and Data Foundations for ML

Every code example from Chapter 10, in the same order as the book. Run the set-up cell first (needs an internet connection).

## Set-up: load the data and keep the missing values

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

bank_data = fetch_ucirepo(id=222)
bank = bank_data.data.features.copy()
bank["subscribed"] = bank_data.data.targets["y"]
bank = bank.replace("unknown", np.nan)
print(bank.shape)

## 1. First look

In [ ]:
bank.info()

## 2. Missing values

In [ ]:
missing = bank.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
report = pd.DataFrame({"missing": missing, "percent": (missing / len(bank) * 100).round(1)})
print(report)

In [ ]:
for col in ["job", "education", "contact"]:
    rate_missing = (bank.loc[bank[col].isna(), "subscribed"] == "yes").mean()
    rate_present = (bank.loc[bank[col].notna(), "subscribed"] == "yes").mean()
    print(f"{col:<10} missing: {rate_missing:.1%}   present: {rate_present:.1%}")

In [ ]:
print("pdays = -1:      ", (bank["pdays"] == -1).sum())
print("previous = 0:    ", (bank["previous"] == 0).sum())
print("poutcome missing:", bank["poutcome"].isna().sum())
print(bank.loc[(bank["pdays"] != -1) & bank["poutcome"].isna(), ["pdays", "previous", "poutcome"]])

In [ ]:
clean = bank.copy()

clean["contacted_before"] = (clean["pdays"] != -1).astype(int)
clean["poutcome"] = np.where(clean["contacted_before"] == 0, "not_contacted",
                             clean["poutcome"].fillna("unknown"))
clean["contact"] = clean["contact"].fillna("unknown")
clean["education"] = clean["education"].fillna("unknown")
clean["job"] = clean["job"].fillna(clean["job"].mode()[0])

print(clean["poutcome"].value_counts())
print("Missing values left:", clean.isna().sum().sum())

## 3. Duplicates

In [ ]:
print("Exact duplicate rows:", clean.duplicated().sum())

profile = ["age", "job", "marital", "education", "balance", "housing", "loan"]
shared = clean.duplicated(subset=profile, keep=False)
print("Rows that share a profile with another row:", shared.sum())
print(clean.loc[shared].sort_values(profile).head(4)[profile + ["month", "subscribed"]])

## 4. Outliers

In [ ]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["age", "balance", "campaign"]:
    low, high = iqr_bounds(clean[col])
    outside = ((clean[col] < low) | (clean[col] > high)).sum()
    print(f"{col:<9} normal range {low:>8.1f} to {high:>7.1f}   outliers: {outside}")

In [ ]:
print("Oldest customers:", clean["age"].sort_values(ascending=False).head(5).tolist())
print("Most calls:      ", clean["campaign"].sort_values(ascending=False).head(5).tolist())

cap = clean["campaign"].quantile(0.99)
clean["campaign_capped"] = clean["campaign"].clip(upper=cap)
print("99th percentile of campaign:", cap)
print("Max after capping:", clean["campaign_capped"].max())

In [ ]:
clean["balance_log"] = np.sign(clean["balance"]) * np.log1p(clean["balance"].abs())

print("Skew of balance:     ", round(clean["balance"].skew(), 2))
print("Skew of balance_log: ", round(clean["balance_log"].skew(), 2))
print(clean[["balance", "balance_log"]].head(3).round(2))

## 5. Messy text and wrong types

In [ ]:
messy = pd.DataFrame({
    "age": [58, 44, 33, 470, 33, 35],
    "job": ["Management ", "technician", "ENTREPRENEUR", "blue-collar", None, " Technician"],
    "balance": ["2,143", "29", "2", "1,506", "1", "€231"],
    "housing": ["yes", "Yes", "Y", "no", "yes", "NO"],
})
messy = pd.concat([messy, messy.iloc[[1]]], ignore_index=True)
print(messy)
print(messy.dtypes)

In [ ]:
tidy = messy.drop_duplicates().copy()
tidy["job"] = tidy["job"].str.strip().str.lower()
tidy["balance"] = tidy["balance"].str.replace(",", "").str.replace("€", "").astype(int)
tidy["housing"] = tidy["housing"].str.lower().str[0].map({"y": "yes", "n": "no"})
tidy.loc[tidy["age"] > 100, "age"] = np.nan
tidy["age"] = tidy["age"].fillna(tidy["age"].median())

print(tidy)
print(tidy.dtypes)

## 6. Feature engineering

In [ ]:
month_order = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
               "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
clean["month_num"] = clean["month"].map(month_order)
clean["has_any_loan"] = ((clean["housing"] == "yes") | (clean["loan"] == "yes")).astype(int)

print(clean[["month", "month_num", "housing", "loan", "has_any_loan"]].head(3))
print(clean.groupby("has_any_loan")["subscribed"].apply(lambda s: (s == "yes").mean()).round(3))

## 7. Encoding categories

In [ ]:
for col in ["default", "housing", "loan", "subscribed"]:
    clean[col] = clean[col].map({"yes": 1, "no": 0})

sizes = pd.Series(["small", "large", "medium", "small"])
print("Ordinal example:", sizes.map({"small": 1, "medium": 2, "large": 3}).tolist())

marital_dummies = pd.get_dummies(clean["marital"], prefix="marital", dtype=int)
print(pd.concat([clean["marital"], marital_dummies], axis=1).head(4))

## 8. Scaling numbers

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

cols = ["age", "balance_log", "campaign_capped"]
standard = StandardScaler().fit_transform(clean[cols])
minmax = MinMaxScaler().fit_transform(clean[cols])

summary = ["mean", "std", "min", "max"]
print("StandardScaler:")
print(pd.DataFrame(standard, columns=cols).describe().loc[summary].round(2) + 0.0)
print("MinMaxScaler:")
print(pd.DataFrame(minmax, columns=cols).describe().loc[summary].round(2) + 0.0)

## Mini project: a reusable cleaning pipeline and an EDA report

In [ ]:
def clean_bank(raw):
    """Turn the raw bank table into a numeric, model-ready table."""
    df = raw.copy()

    # 1. Missing values
    df["contacted_before"] = (df["pdays"] != -1).astype(int)
    df["poutcome"] = np.where(df["contacted_before"] == 0, "not_contacted",
                              df["poutcome"].fillna("unknown"))
    df["contact"] = df["contact"].fillna("unknown")
    df["education"] = df["education"].fillna("unknown")
    df["job"] = df["job"].fillna(df["job"].mode()[0])

    # 2. Outliers and skew
    df["campaign"] = df["campaign"].clip(upper=df["campaign"].quantile(0.99))
    df["balance"] = np.sign(df["balance"]) * np.log1p(df["balance"].abs())

    # 3. New features
    month_order = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
                   "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
    df["month"] = df["month"].map(month_order)

    # 4. Encoding
    for col in ["default", "housing", "loan", "subscribed"]:
        df[col] = df[col].map({"yes": 1, "no": 0})
    df = pd.get_dummies(df, columns=["job", "marital", "education", "contact", "poutcome"], dtype=int)

    # 5. Remove leakage
    df = df.drop(columns=["duration"])
    return df


model_ready = clean_bank(bank)
print("Shape:", model_ready.shape)
print("Missing values:", model_ready.isna().sum().sum())
print("Non-numeric columns:", model_ready.select_dtypes(exclude="number").columns.tolist())
print("Raw table unchanged:", bank.shape, bank["job"].isna().sum(), "missing jobs")
model_ready.to_csv("bank_model_ready.csv", index=False)

In [ ]:
rate_by_month = model_ready.groupby("month")["subscribed"].mean() * 100
poutcome_cols = [c for c in model_ready.columns if c.startswith("poutcome_")]
rate_by_poutcome = pd.Series(
    {c.replace("poutcome_", ""): model_ready.loc[model_ready[c] == 1, "subscribed"].mean() * 100
     for c in poutcome_cols}
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(rate_by_month.index, rate_by_month.values)
axes[0].set_xticks(range(1, 13))
axes[0].set_title("Subscription rate (%) by month")
axes[0].set_xlabel("Month (1 = January)")
axes[1].barh(rate_by_poutcome.index, rate_by_poutcome.values)
axes[1].set_title("Subscription rate (%) by previous campaign outcome")
fig.tight_layout()
plt.show()

print(rate_by_month.round(1).to_dict())
print(rate_by_poutcome.round(1).to_dict())
print(bank["month"].value_counts().head(3).to_dict())

## Try it yourself
Use the empty cells below for the exercises at the end of Chapter 10.

---
**Dataset credit:** S. Moro, P. Rita and P. Cortez, *Bank Marketing* dataset, UCI Machine Learning Repository (2014), https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.